In [ ]:
%pip install -U langchain-core langchain langchain-community langchain-groq langgraph

## Implementation
This notebook implements an autonomous AI email agent utilizing LangChain, LangGraph, and the Groq LLM. The system defines two primary tools using Python's standard `imaplib` and `smtplib` libraries: one for securely fetching and parsing unread emails via IMAP, and another for transmitting responses via SMTP. By employing a ReAct (Reasoning and Acting) architecture, the agent can autonomously execute natural language instructions to read incoming messages, comprehend their context, generate formal, context-aware replies, and dispatch them without manual intervention.

In [ ]:
import os
import smtplib
import imaplib
import email
from email.mime.text import MIMEText
from email.header import decode_header
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langgraph.prebuilt import create_react_agent

os.environ["GROQ_API_KEY"] = "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"

MY_EMAIL = "email_address"  
MY_PASSWORD = "app_password" 

@tool
def check_unread_emails_gmail() -> str:
    """
    Use this tool to read the latest UNREAD email from the Gmail inbox.
    Returns the sender, subject, and body of the email.
    """
    print("\n Agent is securely connecting to Gmail to read emails...")
    try:
        mail = imaplib.IMAP4_SSL("imap.gmail.com", 993)
        mail.login(MY_EMAIL, MY_PASSWORD)
        mail.select("inbox")

        status, messages = mail.search(None, "UNSEEN")
        if status != "OK" or not messages[0]:
            return "No new unread emails found."

        email_ids = messages[0].split()
        latest_email_id = email_ids[-1]

        status, msg_data = mail.fetch(latest_email_id, "(RFC822)")
        email_content = ""
        
        for response_part in msg_data:
            if isinstance(response_part, tuple):
                msg = email.message_from_bytes(response_part[1])
                
                subject, encoding = decode_header(msg["Subject"])[0]
                if isinstance(subject, bytes):
                    subject = subject.decode(encoding if encoding else "utf-8")
                
                sender = msg.get("From")
                
                body = ""
                if msg.is_multipart():
                    for part in msg.walk():
                        if part.get_content_type() == "text/plain":
                            body = part.get_payload(decode=True).decode("utf-8", errors="ignore")
                            break
                else:
                    body = msg.get_payload(decode=True).decode("utf-8", errors="ignore")
                body = body[:2000] + "\n...(Text shortened due to length)"
                
                email_content = f"From: {sender}\nSubject: {subject}\nBody:\n{body}"
        
        mail.logout()
        return email_content

    except Exception as e:
        return f"Failed to read emails from Gmail. Error: {e}"

@tool
def send_email_gmail(to_address: str, subject: str, body: str) -> str:
    """
    Use this tool to send an email via Gmail. 
    Requires exactly 3 string arguments: to_address, subject, and body.
    """
    print(f"\n Agent is sending email to {to_address} via Gmail...")
    try:
        msg = MIMEText(body)
        msg['Subject'] = subject
        msg['From'] = MY_EMAIL
        msg['To'] = to_address

        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp_server:
            smtp_server.login(MY_EMAIL, MY_PASSWORD)
            smtp_server.sendmail(MY_EMAIL, to_address, msg.as_string())
        
        return "Success: Email was sent successfully via Gmail!"
    except Exception as e:
        return f"Failed to send email. Error: {e}"

llm = ChatGroq(temperature=0, model_name="openai/gpt-oss-120b")
tools = [check_unread_emails_gmail, send_email_gmail]
email_agent = create_react_agent(llm, tools)

print(" AI Agent is Ready!")

In [ ]:
task = """
۱. وارد اینباکس من شو و آخرین ایمیل خوانده نشده را پیدا کن.
۲. ایمیل را دقیق بخوان. متوجه شو که نویسنده چه چیزی از من خواسته است.
۳.  یک جواب کامل در چندخط رسمی و محترمانه (به زبان فارسی) تولید کن که می‌گویدکه درخواست ملاقات را قبول می کند."
۴. آن جواب تولید شده را به ایمیل فرستنده ریپلای (ارسال) کن.
حتما زیر این ایمیل بنویس که متن توسط هوش مصنوعی تولید و ارسال شده است
"""

print(f"You: {task}\n")

response = email_agent.invoke({"messages": [("user", task)]})

print("\n Agent Final Report:")
print(response["messages"][-1].content)